In [1]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

In [2]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None)}

In [3]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", climate_scen)
    circular_economy_scenario_dirs = None

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs) 


    factory = ModelFactory(
    [bld_sector], complete_timeline
    ).add(GenericStocks, ["buildings"]
    ).add(MaterialIntensities, "buildings",
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    print(f"Finished {scen_id}")

C:\Coding\image-materials\imagematerials\buildings\preprocessing\main.py:78: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'material' ('material',) The recommendation is to set join explicitly for this case.
  mat_intensities = xr.concat((mat_intensities_res, mat_intensities_comm), dim="Type")


KeyError: "not all values found in index 'ScipyParam'. Try setting the `method` keyword argument (example: method='nearest')."

In [4]:
bld_sector.prep_data.keys()



dict_keys(['stocks', 'lifetimes', 'material_intensities', 'knowledge_graph', 'set_unit_flexible'])

In [7]:
bld_sector.prep_data.get("lifetimes").get("weibull")

<xarray.DataArray (Time: 380, Region: 26, Type: 12, ScipyParam: 2)> Size: 2MB
<Quantity([[[[ 1.443      49.567     ]
   [ 1.443      49.567     ]
   [ 1.443      49.567     ]
   ...
   [ 1.96820464 57.52862846]
   [ 1.96820464 57.52862846]
   [ 1.96820464 57.52862846]]

  [[ 1.443      49.567     ]
   [ 1.443      49.567     ]
   [ 1.443      49.567     ]
   ...
   [ 4.16343417 85.18683893]
   [ 4.16343417 85.18683893]
   [ 4.16343417 85.18683893]]

  [[ 1.443      49.567     ]
   [ 1.443      49.567     ]
   [ 1.443      49.567     ]
   ...
...
   ...
   [ 1.96820464 94.00102689]
   [ 1.96820464 94.00102689]
   [ 1.96820464 94.00102689]]

  [[ 1.443      49.567     ]
   [ 1.443      49.567     ]
   [ 1.443      49.567     ]
   ...
   [ 1.96820464 67.35799478]
   [ 1.96820464 67.35799478]
   [ 1.96820464 67.35799478]]

  [[ 1.443      49.567     ]
   [ 1.443      49.567     ]
   [ 1.443      49.567     ]
   ...
   [ 1.96820464 67.35799478]
   [ 1.96820464 67.35799478]
   [ 1.96820464 67.35799478]]]], 'dimensionless')>
Coordinates:
  * Type        (Type) object 96B 'Office' 'Retail+' ... 'Semi-detached - Urban'
  * Time        (Time) int64 3kB 1721 1722 1723 1724 ... 2097 2098 2099 2100
  * ScipyParam  (ScipyParam) <U5 40B 'c' 'scale'
  * Region      (Region) <U5 520B 'CAN' 'USA' 'MEX' ... 'OCE' 'RSAS' 'RSAF'

In [ ]:
import matplotlib.pyplot as plt


bld_sector.prep_data.get("stocks").sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

bld_sector.prep_data.get("stocks").sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')
plt.legend()

In [ ]:
model.buildings.keys()

In [ ]:
model.buildings.get("inflow").to_array().sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow").to_array().sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array().sel(material = 'Steel', Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow_materials").to_array().sel(material = 'Steel', Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.save_pkl("test.pkl")
model2 = ModelFactory.load_pkl("test.pkl")
model2.buildings["inflow_materials"]